# Credit Scorecard

In [2]:
import pandas as pd
import numpy as np

In [3]:
# 1. Load and clean
loan_train = pd.read_csv('./Data/train.csv')



In [4]:
clean_age = loan_train.loc[loan_train['person_age'] <= 100, 'person_age'].median()
clean_emp = loan_train.loc[loan_train['person_emp_length'] <= 60, 'person_emp_length'].median()
loan_train.loc[loan_train['person_age'] > 100, 'person_age'] = clean_age
loan_train.loc[loan_train['person_emp_length'] > 60, 'person_emp_length'] = clean_emp

In [5]:
# Test Data
test_df = pd.read_csv('./Data/test.csv')
test_ids = test_df['id']
test_loan = test_df.drop(columns=['id'])


In [6]:
clean_age = test_loan.loc[test_loan['person_age'] <= 100, 'person_age'].median()
clean_emp = test_loan.loc[test_loan['person_emp_length'] <= 60, 'person_emp_length'].median()
test_loan.loc[test_loan['person_age'] > 100, 'person_age'] = clean_age
test_loan.loc[test_loan['person_emp_length'] > 60, 'person_emp_length'] = clean_emp

In [7]:

# STEP 1: BIN THE CONTINUOUS VARIABLES

def create_bins(df, column, n_bins=5):
    """
    Splits a continuous column into n_bins equal-sized groups (by quantile),
    which is the standard approach in credit scorecards.
    """
    df[f'{column}_bin'] = pd.qcut(df[column], q=n_bins, duplicates='drop')
    return df




In [8]:
# Run it for every binned feature and collect the results
feature_binning = [
    'person_age', 'person_income', 'loan_int_rate',
    'loan_percent_income', 'person_emp_length', 'cb_person_cred_hist_length'
]

In [11]:

for feature in feature_binning:
    loan_train = create_bins(loan_train, feature, n_bins=5)
    test_loan = create_bins(test_loan, feature, n_bins=5)

'''# Apply binning to the key continuous variables
loan_train = create_bins(loan_train, 'person_age', n_bins=5)
loan_train = create_bins(loan_train, 'person_income', n_bins=5)
loan_train = create_bins(loan_train, 'loan_int_rate', n_bins=5)
loan_train = create_bins(loan_train, 'loan_percent_income', n_bins=5)
loan_train = create_bins(loan_train, 'person_emp_length', n_bins=5)
loan_train = create_bins(loan_train, 'cb_person_cred_hist_length', n_bins=5)
'''
print(loan_train[['person_age', 'person_age_bin']].head())

print(test_loan[['person_income', 'person_income_bin']].head())


   person_age  person_age_bin
0          37    (31.0, 84.0]
1          22  (19.999, 23.0]
2          29    (27.0, 31.0]
3          30    (27.0, 31.0]
4          22  (19.999, 23.0]
   person_income     person_income_bin
0          69000    (63000.0, 83000.0]
1          96000  (83000.0, 1900000.0]
2          30000   (3999.999, 40000.0]
3          50000    (40000.0, 50000.0]
4         102000  (83000.0, 1900000.0]


In [12]:

# STEP 2: CALCULATE WOE AND IV PER BIN

def calculate_woe_iv(df, feature, target):
    
    grouped = df.groupby(feature)[target].agg(['count', 'sum'])
    grouped.columns = ['total', 'bad']       # 'bad' = defaulted (loan_status = 1)
    grouped['good'] = grouped['total'] - grouped['bad']

    total_bad = grouped['bad'].sum()
    total_good = grouped['good'].sum()

    # Add a tiny value to avoid division by zero if a bin has 0 defaults
    grouped['bad_pct'] = (grouped['bad'] + 0.5) / (total_bad + 0.5)
    grouped['good_pct'] = (grouped['good'] + 0.5) / (total_good + 0.5)

    grouped['woe'] = np.log(grouped['good_pct'] / grouped['bad_pct'])
    grouped['iv_contribution'] = (grouped['good_pct'] - grouped['bad_pct']) * grouped['woe']

    iv_total = grouped['iv_contribution'].sum()

    return grouped, iv_total


In [13]:

# Run it for every binned feature and collect the results
binned_features = [
    'person_age_bin', 'person_income_bin', 'loan_int_rate_bin',
    'loan_percent_income_bin', 'person_emp_length_bin', 'cb_person_cred_hist_length_bin'
]

# Also include the existing categorical columns - they don't need binning
categorical_features = ['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']



In [14]:


iv_summary = {}

print("\n===== WOE / IV BY FEATURE =====\n")
for feature in binned_features + categorical_features:
    woe_table, iv = calculate_woe_iv(loan_train, feature, 'loan_status')
    iv_summary[feature] = iv
    print(f"--- {feature} (IV = {iv:.4f}) ---")
    print(woe_table[['total', 'bad', 'good', 'woe']].round(3))
    print()

# Rank features by predictive power - this tells you what actually drives default risk
iv_ranking = pd.Series(iv_summary).sort_values(ascending=False)
print("===== FEATURE RANKING BY INFORMATION VALUE =====")
print(iv_ranking.round(4))


===== WOE / IV BY FEATURE =====

--- person_age_bin (IV = 0.0037) ---
                total   bad   good    woe
person_age_bin                           
(19.999, 23.0]  16584  2507  14077 -0.070
(23.0, 25.0]    11462  1648   9814 -0.012
(25.0, 27.0]     8325  1103   7222  0.083
(27.0, 31.0]    11227  1491   9736  0.081
(31.0, 84.0]    11047  1601   9446 -0.021

--- person_income_bin (IV = 0.4981) ---
                      total   bad   good    woe
person_income_bin                              
(4199.999, 39480.0]   11731  3559   8172 -0.964
(39480.0, 50000.0]    12199  1656  10543  0.055
(50000.0, 63000.0]    11385  1519   9866  0.075
(63000.0, 83119.2]    11601  1100  10501  0.460
(83119.2, 1900000.0]  11729   516  11213  1.282

--- loan_int_rate_bin (IV = 0.9272) ---
                   total   bad   good    woe
loan_int_rate_bin                           
(5.419, 7.51]      13264   557  12707  1.331
(7.51, 9.99]       10741   730  10011  0.822
(9.99, 11.49]      11850  1162  10688

In [15]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression

# ============================================
# STEP 3: TRANSFORM TRAIN DATA - REPLACE RAW VALUES WITH WOE
# ============================================

def apply_woe(df, feature, woe_table):
    """
    Maps each row's bin to its WOE value.
    woe_table is the 'grouped' table returned by calculate_woe_iv() for this feature.
    """
    woe_map = woe_table['woe'].to_dict()
    return df[feature].map(woe_map).astype(float)

# Rebuild the WOE tables and store them - we need to reuse the SAME tables on test data later
woe_tables = {}
all_features = binned_features + categorical_features

for feature in all_features:
    woe_table, iv = calculate_woe_iv(loan_train, feature, 'loan_status')
    woe_tables[feature] = woe_table

# Build a new dataframe of WOE-transformed features
X_woe_train = pd.DataFrame()
for feature in all_features:
    new_col_name = feature.replace('_bin', '') + '_woe'
    X_woe_train[new_col_name] = apply_woe(loan_train, feature, woe_tables[feature])

y_train_full = loan_train['loan_status']

print(X_woe_train.head())
print(X_woe_train.isnull().sum())  # check nothing failed to map


# ============================================
# STEP 4: FIT LOGISTIC REGRESSION ON WOE VALUES
# ============================================

scorecard_model = LogisticRegression(class_weight='balanced', random_state=42)
scorecard_model.fit(X_woe_train, y_train_full)

print('\nCoefficients:')
for feature, coef in zip(X_woe_train.columns, scorecard_model.coef_[0]):
    print(f'{feature}: {coef:.4f}')
print(f'Intercept: {scorecard_model.intercept_[0]:.4f}')


# ============================================
# STEP 5: CONVERT TO A POINTS-BASED SCORECARD
# ============================================

# Standard scorecard scaling: pick a target score and how many points double the odds
# These are the industry-conventional defaults (same ones FICO-style scores use)
base_score = 600
pdo = 20  # "points to double the odds"
factor = pdo / np.log(2)
offset = base_score - factor * scorecard_model.intercept_[0]

print(f'\nFactor: {factor:.2f}, Offset: {offset:.2f}')

# Build a points table: for every bin of every feature, how many points it's worth
scorecard_points = []
for feature, coef in zip(X_woe_train.columns, scorecard_model.coef_[0]):
    original_feature = [f for f in all_features if f.replace('_bin', '') == feature.replace('_woe', '')][0]
    woe_table = woe_tables[original_feature]
    for bin_label, row in woe_table.iterrows():
        points = -(coef * row['woe']) * factor
        scorecard_points.append({
            'feature': feature,
            'bin': bin_label,
            'woe': round(row['woe'], 3),
            'points': round(points, 1)
        })

scorecard_df = pd.DataFrame(scorecard_points)
print('\n===== SCORECARD =====')
print(scorecard_df.to_string(index=False))


# ============================================
# STEP 6: APPLY THE SAME BINS + WOE TO TEST DATA
# ============================================

def apply_same_bins(df, column, train_bin_edges):
    """Applies the SAME bin edges learned from train onto test - never re-bin test independently."""
    return pd.cut(df[column], bins=train_bin_edges, include_lowest=True)

# Get the bin edges pandas used on train, so test uses IDENTICAL cut points
bin_edges = {}
for col in ['person_age', 'person_income', 'loan_int_rate', 'loan_percent_income',
            'person_emp_length', 'cb_person_cred_hist_length']:
    _, edges = pd.qcut(loan_train[col], q=5, duplicates='drop', retbins=True)
    bin_edges[col] = edges

test_loan_binned = test_loan.copy()
for col, edges in bin_edges.items():
    test_loan_binned[f'{col}_bin'] = apply_same_bins(test_loan_binned, col, edges)

# Transform test data into WOE using train's WOE tables (never recalculate WOE from test)
X_woe_test = pd.DataFrame()
for feature in all_features:
    new_col_name = feature.replace('_bin', '') + '_woe'
    X_woe_test[new_col_name] = apply_woe(test_loan_binned, feature, woe_tables[feature])

# Fill any bin that didn't exist in train (e.g. a test value outside train's range) with 0 (neutral WOE)
X_woe_test = X_woe_test.fillna(0)


# ============================================
# STEP 7: PREDICT ON TEST AND SAVE TO CSV
# ============================================

test_predictions = scorecard_model.predict(X_woe_test)
test_probabilities = scorecard_model.predict_proba(X_woe_test)[:, 1]

results = pd.DataFrame({
    'id': test_ids,
    'loan_status': test_predictions,
    'default_probability': test_probabilities.round(4)
})

results.to_csv('logisticregression_scorecard_test.csv', index=False)
print(f'\nSaved {len(results)} predictions to logisticregression_scorecard_test.csv')
print(f'Predicted default rate: {test_predictions.mean() * 100:.2f}%')

   person_age_woe  person_income_woe  loan_int_rate_woe  \
0       -0.020891          -0.964439           0.423001   
1       -0.070303           0.075170           0.098691   
2        0.080505          -0.964439           0.822166   
3        0.080505           0.460159           0.423001   
4       -0.070303           0.075170           1.330891   

   loan_percent_income_woe  person_emp_length_woe  \
0                 0.535192              -0.444359   
1                 0.873921               0.257940   
2                 0.214743               0.491534   
3                 0.535192               0.491534   
4                 0.726368              -0.167445   

   cb_person_cred_hist_length_woe  person_home_ownership_woe  loan_intent_woe  \
0                       -0.005109                  -0.544837         0.318176   
1                       -0.045714                   2.469350        -0.267555   
2                       -0.005109                   2.469350         0.080621   
3 

In [16]:
scorecard_df.to_csv('credit_scorecard.csv', index=False)
print(f'Saved scorecard with {len(scorecard_df)} rows to credit_scorecard.csv')

Saved scorecard with 49 rows to credit_scorecard.csv
